# Real-Time Customer Support Intelligence Platform
## Modern Data Engineering for AI Systems — Capstone Project

This notebook implements the selected capstone idea: **Real-Time Customer Support Intelligence Platform**.

The project is organized according to the five required deliverables:

| Deliverable | Requirement | Implementation |
|---|---|---|
| 1 | Ingestion layer | Mock Kafka producer + consumer + Pydantic validation |
| 2 | Delta Lakehouse | Bronze/Silver/Gold simulation + schema enforcement + MERGE |
| 3 | RAG pipeline | Chunking + embeddings + vector search + BM25 + RRF + reranking |
| 4 | Orchestration | Airflow-style DAG + reference Airflow DAG file |
| 5 | Quality + lineage | Great Expectations-style suite + OpenLineage-style events |

## Project Design Note

This project is intentionally lightweight and runnable in local JupyterLab.

Production tools such as Kafka, Delta Lake, Airflow, Great Expectations, and OpenLineage are represented using educational simulations where a full local deployment would be too heavy.  
The RAG pipeline uses real ML retrieval components.

In [ ]:
# Setup — install required packages
import sys

!{sys.executable} -m pip install pandas numpy pyarrow pydantic loguru kagglehub sentence-transformers scikit-learn rank-bm25 chromadb -q

In [1]:
# Common imports and project paths

import os
import json
import uuid
import re
from pathlib import Path
from datetime import datetime, UTC
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
import kagglehub

from pydantic import BaseModel, Field, ValidationError
from loguru import logger

# Avoid duplicate Loguru handlers when re-running the notebook
logger.remove()
logger.add("capstone_run.log", rotation="10 MB", level="INFO")

BASE_DIR = Path.cwd()

INGESTION_PATH = BASE_DIR / "outputs" / "ingestion"
LINEAGE_PATH = BASE_DIR / "outputs" / "lineage"

INGESTION_PATH.mkdir(parents=True, exist_ok=True)
LINEAGE_PATH.mkdir(parents=True, exist_ok=True)

VALID_EVENTS_FILE = INGESTION_PATH / "valid_events.jsonl"
INVALID_EVENTS_FILE = INGESTION_PATH / "invalid_events.jsonl"

print("Project base directory:", BASE_DIR)

Project base directory: C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project


C:\Users\alamr\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
KAGGLE_DATASET = "suraj520/customer-support-ticket-dataset"

dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
csv_files = list(Path(dataset_path).glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV file found in Kaggle dataset.")

df_raw = pd.read_csv(csv_files[0])

print("Dataset loaded successfully")
print("Shape:", df_raw.shape)
df_raw.head()

Dataset loaded successfully
Shape: (8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
# Dataset overview

dataset_profile = {
    "rows": df_raw.shape[0],
    "columns": df_raw.shape[1],
    "duplicate_rows": int(df_raw.duplicated().sum()),
    "columns_list": df_raw.columns.tolist()
}

missing_values = df_raw.isnull().sum().sort_values(ascending=False)

print("Dataset profile:")
print(dataset_profile)

print("\nTop missing-value columns:")
display(missing_values.head(10))

print("\nTicket Priority distribution:")
display(df_raw["Ticket Priority"].value_counts())

df_raw.head()

Dataset profile:
{'rows': 8469, 'columns': 17, 'duplicate_rows': 0, 'columns_list': ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']}

Top missing-value columns:


Customer Satisfaction Rating    5700
Resolution                      5700
Time to Resolution              5700
First Response Time             2819
Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Ticket Subject                     0
dtype: int64


Ticket Priority distribution:


Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


# Deliverable 1 — Ingestion Layer
## Kafka producer + consumer with schema validation

This module implements the ingestion layer for customer support tickets.

**What it demonstrates:**

- A Kafka-style producer that streams ticket rows as events.
- A Kafka-style consumer that reads messages from the raw topic.
- Pydantic schema validation for incoming events.
- Separation of valid and invalid records into JSONL files.

**Outputs:**

- `outputs/ingestion/valid_events.jsonl`
- `outputs/ingestion/invalid_events.jsonl`

In [4]:
class SupportTicketEvent(BaseModel):
    ticket_id: str = Field(..., min_length=1)
    ticket_subject: str = Field(..., min_length=3)
    ticket_description: str = Field(..., min_length=10)
    ticket_type: str = Field(..., min_length=2)
    ticket_status: str = Field(..., min_length=2)
    resolution: str = Field(..., min_length=3)
    ticket_priority: str = Field(..., min_length=2)
    ticket_channel: str = Field(..., min_length=2)
    product_purchased: str | None = None
    date_of_purchase: str | None = None

In [5]:
class MockKafka:
    """
    Simulates Kafka topic / offset behavior.
    Similar to the trainer's capstone code.
    """

    def __init__(self):
        self.topics = defaultdict(list)
        self.offsets = defaultdict(int)

    def produce(self, topic: str, event: dict):
        self.topics[topic].append({
            "offset": self.offsets[topic],
            "event_time": datetime.now(UTC).isoformat(),
            "payload": event
        })
        self.offsets[topic] += 1

    def consume_all(self, topic: str) -> list[dict]:
        return self.topics.pop(topic, [])

In [6]:
RAW_TOPIC = "support_tickets_raw"


def normalize_ticket(row: pd.Series) -> dict[str, Any]:
    return {
        "ticket_id": str(row["Ticket ID"]),
        "ticket_subject": str(row["Ticket Subject"]),
        "ticket_description": str(row["Ticket Description"]),
        "ticket_type": str(row["Ticket Type"]),
        "ticket_status": str(row["Ticket Status"]),
        "resolution": str(row["Resolution"]),
        "ticket_priority": str(row["Ticket Priority"]),
        "ticket_channel": str(row["Ticket Channel"]),
        "product_purchased": str(row["Product Purchased"]),
        "date_of_purchase": str(row["Date of Purchase"]),
    }


def module1_ingest(kafka: MockKafka, df: pd.DataFrame, limit: int = 50) -> int:
    """
    MODULE 1 — INGESTION

    Reads Customer Support Ticket Dataset from Kaggle and streams
    each row as an event into a Kafka-like topic.
    """

    logger.info("[MODULE 1] Ingestion started")

    produced_count = 0

    for _, row in df.head(limit).iterrows():
        event = normalize_ticket(row)
        kafka.produce(RAW_TOPIC, event)
        produced_count += 1

    logger.info(f"[MODULE 1] Produced {produced_count} support ticket events")
    return produced_count

In [7]:
def write_jsonl(path: Path, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def module1_consume_and_validate(kafka: MockKafka) -> dict:
    """
    Consumes events from Mock Kafka and validates each ticket
    against the Pydantic data contract.
    """

    logger.info("[MODULE 1] Consumer validation started")

    if VALID_EVENTS_FILE.exists():
        VALID_EVENTS_FILE.unlink()

    if INVALID_EVENTS_FILE.exists():
        INVALID_EVENTS_FILE.unlink()

    messages = kafka.consume_all(RAW_TOPIC)

    valid_count = 0
    invalid_count = 0

    for msg in messages:
        raw_event = msg["payload"]

        try:
            validated = SupportTicketEvent(**raw_event)

            output_record = {
                "kafka_metadata": {
                    "topic": RAW_TOPIC,
                    "offset": msg["offset"],
                    "event_time": msg["event_time"]
                },
                "event": validated.model_dump()
            }

            write_jsonl(VALID_EVENTS_FILE, output_record)
            valid_count += 1

        except ValidationError as e:
            output_record = {
                "kafka_metadata": {
                    "topic": RAW_TOPIC,
                    "offset": msg["offset"],
                    "event_time": msg["event_time"]
                },
                "raw_event": raw_event,
                "validation_errors": e.errors()
            }

            write_jsonl(INVALID_EVENTS_FILE, output_record)
            invalid_count += 1

    logger.info(f"[MODULE 1] Valid={valid_count}, Invalid={invalid_count}")

    return {
        "valid_events": valid_count,
        "invalid_events": invalid_count,
        "valid_path": str(VALID_EVENTS_FILE),
        "invalid_path": str(INVALID_EVENTS_FILE)
    }

In [8]:
kafka = MockKafka()

produced = module1_ingest(kafka, df_raw, limit=50)
validation_summary = module1_consume_and_validate(kafka)

print("Produced events:", produced)
validation_summary

Produced events: 50


{'valid_events': 50,
 'invalid_events': 0,
 'valid_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\valid_events.jsonl',
 'invalid_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\invalid_events.jsonl'}

In [9]:
with open(VALID_EVENTS_FILE, "r", encoding="utf-8") as f:
    sample_valid_event = json.loads(f.readline())

sample_valid_event

{'kafka_metadata': {'topic': 'support_tickets_raw',
  'offset': 0,
  'event_time': '2026-07-04T10:39:27.586498+00:00'},
 'event': {'ticket_id': '1',
  'ticket_subject': 'Product setup',
  'ticket_description': "I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",
  'ticket_type': 'Technical issue',
  'ticket_status': 'Pending Customer Response',
  'resolution': 'nan',
  'ticket_priority': 'Critical',
  'ticket_channel': 'Social media',
  'product_purchased': 'GoPro Hero',
  'date_of_purchase': '2021-03-22'}}

# Deliverable 2 — Delta Lakehouse Simulation
## Bronze / Silver / Gold zones with schema enforcement and MERGE

This module implements a lightweight Delta Lakehouse simulation using Parquet files.

Apache Spark Delta Lake was not used in this local Windows notebook because it requires a working Java/Spark runtime.  
However, the implementation still demonstrates the required Lakehouse concepts:

- Bronze zone for raw validated events.
- Silver zone for cleaned and standardized records.
- Gold zone for curated RAG-ready knowledge documents.
- Schema enforcement before downstream use.
- MERGE / UPSERT behavior for updates and inserts.

In [10]:
BRONZE_PATH = BASE_DIR / "outputs" / "bronze"
SILVER_PATH = BASE_DIR / "outputs" / "silver"
GOLD_PATH = BASE_DIR / "outputs" / "gold"

BRONZE_PATH.mkdir(parents=True, exist_ok=True)
SILVER_PATH.mkdir(parents=True, exist_ok=True)
GOLD_PATH.mkdir(parents=True, exist_ok=True)

BRONZE_FILE = BRONZE_PATH / "support_tickets_bronze.parquet"
SILVER_FILE = SILVER_PATH / "support_tickets_silver.parquet"
GOLD_FILE = GOLD_PATH / "support_knowledge_base_gold.parquet"

print("Lakehouse folders are ready.")

Lakehouse folders are ready.


In [11]:
def module2_bronze_load(valid_events_file: Path) -> pd.DataFrame:
    """
    MODULE 2A — BRONZE LAYER

    Loads validated Kafka-style events from JSONL and stores them
    as raw Bronze data.
    """

    logger.info("[MODULE 2A] Bronze load started")

    records = []

    with open(valid_events_file, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    bronze_rows = []

    for record in records:
        event = record["event"]
        metadata = record["kafka_metadata"]

        bronze_rows.append({
            **event,
            "kafka_topic": metadata["topic"],
            "kafka_offset": metadata["offset"],
            "event_time": metadata["event_time"],
            "ingestion_time": datetime.now(UTC).isoformat()
        })

    bronze_df = pd.DataFrame(bronze_rows)

    bronze_df.to_parquet(BRONZE_FILE, index=False)

    logger.info(f"[MODULE 2A] Bronze saved: {len(bronze_df)} records")

    return bronze_df


bronze_df = module2_bronze_load(VALID_EVENTS_FILE)

print("Bronze shape:", bronze_df.shape)
bronze_df.head()

Bronze shape: (50, 14)


,ticket_id,ticket_subject,ticket_description,ticket_type,ticket_status,resolution,ticket_priority,ticket_channel,product_purchased,date_of_purchase,kafka_topic,kafka_offset,event_time,ingestion_time
0,1,Product setup,I'm having an issue with the {product_purchase...,Technical issue,Pending Customer Response,nan,Critical,Social media,GoPro Hero,2021-03-22,support_tickets_raw,0,2026-07-04T10:39:27.586498+00:00,2026-07-04T10:39:29.917593+00:00
1,2,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,Pending Customer Response,nan,Critical,Chat,LG Smart TV,2021-05-22,support_tickets_raw,1,2026-07-04T10:39:27.586593+00:00,2026-07-04T10:39:29.917609+00:00
2,3,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Closed,Case maybe show recently my computer follow.,Low,Social media,Dell XPS,2020-07-14,support_tickets_raw,2,2026-07-04T10:39:27.586662+00:00,2026-07-04T10:39:29.917615+00:00
3,4,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Closed,Try capital clearly never color toward story.,Low,Social media,Microsoft Office,2020-11-13,support_tickets_raw,3,2026-07-04T10:39:27.586721+00:00,2026-07-04T10:39:29.917621+00:00
4,5,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Closed,West decision evidence bit.,Low,Email,Autodesk AutoCAD,2020-02-04,support_tickets_raw,4,2026-07-04T10:39:27.586780+00:00,2026-07-04T10:39:29.917625+00:00


In [12]:
def clean_text(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in ["nan", "none", "null"]:
        return ""

    return text


def normalize_status(value):
    text = clean_text(value).lower()

    if "closed" in text or "resolved" in text or "solved" in text:
        return "resolved"
    if "pending" in text:
        return "pending"
    if "open" in text:
        return "open"

    return "unknown"


def normalize_channel(value):
    text = clean_text(value).lower()

    if "email" in text:
        return "email"
    if "chat" in text:
        return "chat"
    if "phone" in text:
        return "phone"
    if "social" in text:
        return "social_media"

    return "other"


def normalize_priority(value):
    text = clean_text(value).lower()

    if "critical" in text:
        return "critical"
    if "high" in text:
        return "high"
    if "medium" in text:
        return "medium"
    if "low" in text:
        return "low"

    return "unknown"

In [13]:
def module2_silver_transform(bronze_df: pd.DataFrame) -> pd.DataFrame:
    """
    MODULE 2B — SILVER LAYER

    Cleans raw support tickets and prepares structured data.
    """

    logger.info("[MODULE 2B] Silver transform started")

    silver_df = bronze_df.copy()

    text_columns = [
        "ticket_subject",
        "ticket_description",
        "ticket_type",
        "ticket_status",
        "resolution",
        "ticket_priority",
        "ticket_channel",
        "product_purchased",
    ]

    for col in text_columns:
        silver_df[col] = silver_df[col].apply(clean_text)

    silver_df["status_standardized"] = silver_df["ticket_status"].apply(normalize_status)
    silver_df["channel_standardized"] = silver_df["ticket_channel"].apply(normalize_channel)
    silver_df["priority_standardized"] = silver_df["ticket_priority"].apply(normalize_priority)

    before_dedup = len(silver_df)
    silver_df = silver_df.drop_duplicates(subset=["ticket_id"])
    after_dedup = len(silver_df)

    silver_df["has_resolution"] = silver_df["resolution"].str.len() > 0

    silver_df["support_context"] = (
        "Subject: " + silver_df["ticket_subject"] + "\n"
        "Description: " + silver_df["ticket_description"] + "\n"
        "Product: " + silver_df["product_purchased"] + "\n"
        "Resolution: " + silver_df["resolution"]
    )

    silver_df.to_parquet(SILVER_FILE, index=False)

    logger.info(
        f"[MODULE 2B] Silver saved: {len(silver_df)} records "
        f"(removed duplicates={before_dedup - after_dedup})"
    )

    return silver_df


silver_df = module2_silver_transform(bronze_df)

print("Silver shape:", silver_df.shape)
silver_df[[
    "ticket_id",
    "ticket_subject",
    "status_standardized",
    "priority_standardized",
    "channel_standardized",
    "has_resolution"
]].head()

Silver shape: (50, 19)


,ticket_id,ticket_subject,status_standardized,priority_standardized,channel_standardized,has_resolution
0,1,Product setup,pending,critical,social_media,False
1,2,Peripheral compatibility,pending,critical,chat,False
2,3,Network problem,resolved,low,social_media,True
3,4,Account access,resolved,low,social_media,True
4,5,Data loss,resolved,low,email,True


In [14]:
def module2_gold_create_knowledge_base(silver_df: pd.DataFrame) -> pd.DataFrame:
    """
    MODULE 2C — GOLD LAYER

    Creates a clean support knowledge base for RAG.
    Only tickets with valid resolutions are included.
    """

    logger.info("[MODULE 2C] Gold knowledge base creation started")

    gold_df = silver_df[
        (silver_df["has_resolution"] == True)
        & (silver_df["ticket_subject"].str.len() > 0)
        & (silver_df["ticket_description"].str.len() > 0)
    ].copy()

    gold_df["document_id"] = "KB-" + gold_df["ticket_id"].astype(str)

    gold_df["knowledge_document"] = (
        "Customer Issue:\n"
        + gold_df["ticket_subject"]
        + "\n\nDetails:\n"
        + gold_df["ticket_description"]
        + "\n\nProduct:\n"
        + gold_df["product_purchased"]
        + "\n\nRecommended Resolution:\n"
        + gold_df["resolution"]
    )

    gold_columns = [
        "document_id",
        "ticket_id",
        "ticket_type",
        "priority_standardized",
        "channel_standardized",
        "product_purchased",
        "knowledge_document",
    ]

    gold_df = gold_df[gold_columns]

    gold_df.to_parquet(GOLD_FILE, index=False)

    logger.info(f"[MODULE 2C] Gold saved: {len(gold_df)} knowledge documents")

    return gold_df


gold_df = module2_gold_create_knowledge_base(silver_df)

print("Gold shape:", gold_df.shape)
gold_df.head()

Gold shape: (18, 7)


,document_id,ticket_id,ticket_type,priority_standardized,channel_standardized,product_purchased,knowledge_document
2,KB-3,3,Technical issue,low,social_media,Dell XPS,Customer Issue:\nNetwork problem\n\nDetails:\n...
3,KB-4,4,Billing inquiry,low,social_media,Microsoft Office,Customer Issue:\nAccount access\n\nDetails:\nI...
4,KB-5,5,Billing inquiry,low,email,Autodesk AutoCAD,Customer Issue:\nData loss\n\nDetails:\nI'm ha...
10,KB-11,11,Cancellation request,high,phone,Nintendo Switch,Customer Issue:\nData loss\n\nDetails:\nI'm ha...
11,KB-12,12,Product inquiry,high,chat,Microsoft Xbox Controller,Customer Issue:\nSoftware bug\n\nDetails:\nI'm...


In [15]:
module2_summary = {
    "module": "Storage Layer",
    "bronze_records": len(bronze_df),
    "silver_records": len(silver_df),
    "gold_knowledge_documents": len(gold_df),
    "bronze_path": str(BRONZE_FILE),
    "silver_path": str(SILVER_FILE),
    "gold_path": str(GOLD_FILE),
}

module2_summary

{'module': 'Storage Layer',
 'bronze_records': 50,
 'silver_records': 50,
 'gold_knowledge_documents': 18,
 'bronze_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\bronze\\support_tickets_bronze.parquet',
 'silver_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\silver\\support_tickets_silver.parquet',
 'gold_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\gold\\support_knowledge_base_gold.parquet'}

In [16]:
# MODULE 2D — Schema Enforcement

EXPECTED_SILVER_COLUMNS = {
    "ticket_id",
    "ticket_subject",
    "ticket_description",
    "ticket_type",
    "ticket_status",
    "resolution",
    "ticket_priority",
    "ticket_channel",
    "product_purchased",
    "date_of_purchase",
    "kafka_topic",
    "kafka_offset",
    "event_time",
    "ingestion_time",
    "status_standardized",
    "channel_standardized",
    "priority_standardized",
    "has_resolution",
    "support_context",
}


def module2_schema_enforcement(df: pd.DataFrame, expected_columns: set) -> dict:
    """
    MODULE 2D — SCHEMA ENFORCEMENT

    Simulates Delta Lake schema enforcement by checking that the curated
    Silver table contains all required columns before downstream use.
    """

    logger.info("[MODULE 2D] Schema enforcement started")

    actual_columns = set(df.columns)

    missing_columns = sorted(list(expected_columns - actual_columns))
    unexpected_columns = sorted(list(actual_columns - expected_columns))

    passed = len(missing_columns) == 0

    result = {
        "module": "Schema Enforcement",
        "status": "passed" if passed else "failed",
        "required_columns": sorted(list(expected_columns)),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "checked_at": datetime.now(UTC).isoformat()
    }

    if not passed:
        logger.error(f"[MODULE 2D] Schema enforcement failed: {missing_columns}")
        raise ValueError(result)

    logger.info("[MODULE 2D] Schema enforcement passed")

    return result


schema_enforcement_result = module2_schema_enforcement(
    silver_df,
    EXPECTED_SILVER_COLUMNS
)

schema_enforcement_result

{'module': 'Schema Enforcement',
 'status': 'passed',
 'required_columns': ['channel_standardized',
  'date_of_purchase',
  'event_time',
  'has_resolution',
  'ingestion_time',
  'kafka_offset',
  'kafka_topic',
  'priority_standardized',
  'product_purchased',
  'resolution',
  'status_standardized',
  'support_context',
  'ticket_channel',
  'ticket_description',
  'ticket_id',
  'ticket_priority',
  'ticket_status',
  'ticket_subject',
  'ticket_type'],
 'missing_columns': [],
 'unexpected_columns': [],
 'checked_at': '2026-07-04T10:39:33.213230+00:00'}

In [17]:
# MODULE 2E — Create New Batch for MERGE / UPSERT

new_batch_raw = pd.DataFrame([
    {
        "ticket_id": str(silver_df.iloc[0]["ticket_id"]),
        "ticket_subject": "Updated product setup issue",
        "ticket_description": "Customer provided additional details about the setup issue.",
        "ticket_type": "Technical issue",
        "ticket_status": "Closed",
        "resolution": "Agent provided updated setup instructions and confirmed the issue was resolved.",
        "ticket_priority": "High",
        "ticket_channel": "Email",
        "product_purchased": silver_df.iloc[0]["product_purchased"],
        "date_of_purchase": silver_df.iloc[0]["date_of_purchase"],
        "kafka_topic": RAW_TOPIC,
        "kafka_offset": 9999,
        "event_time": datetime.now(UTC).isoformat(),
        "ingestion_time": datetime.now(UTC).isoformat(),
    },
    {
        "ticket_id": "NEW-999999",
        "ticket_subject": "Refund request for damaged product",
        "ticket_description": "Customer received a damaged product and requests a refund.",
        "ticket_type": "Refund request",
        "ticket_status": "Open",
        "resolution": "Refund request was created and routed to the billing team.",
        "ticket_priority": "Medium",
        "ticket_channel": "Chat",
        "product_purchased": "Wireless Headphones",
        "date_of_purchase": "2024-01-15",
        "kafka_topic": RAW_TOPIC,
        "kafka_offset": 10000,
        "event_time": datetime.now(UTC).isoformat(),
        "ingestion_time": datetime.now(UTC).isoformat(),
    }
])

new_batch_raw

,ticket_id,ticket_subject,ticket_description,ticket_type,ticket_status,resolution,ticket_priority,ticket_channel,product_purchased,date_of_purchase,kafka_topic,kafka_offset,event_time,ingestion_time
0,1,Updated product setup issue,Customer provided additional details about the...,Technical issue,Closed,Agent provided updated setup instructions and ...,High,Email,GoPro Hero,2021-03-22,support_tickets_raw,9999,2026-07-04T10:39:33.885240+00:00,2026-07-04T10:39:33.885252+00:00
1,NEW-999999,Refund request for damaged product,Customer received a damaged product and reques...,Refund request,Open,Refund request was created and routed to the b...,Medium,Chat,Wireless Headphones,2024-01-15,support_tickets_raw,10000,2026-07-04T10:39:33.885259+00:00,2026-07-04T10:39:33.885263+00:00


In [18]:
def prepare_silver_dataframe(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Applies the same Silver transformation logic without saving the output.
    """

    df_temp = df_input.copy()

    text_columns = [
        "ticket_subject",
        "ticket_description",
        "ticket_type",
        "ticket_status",
        "resolution",
        "ticket_priority",
        "ticket_channel",
        "product_purchased",
    ]

    for col in text_columns:
        df_temp[col] = df_temp[col].apply(clean_text)

    df_temp["status_standardized"] = df_temp["ticket_status"].apply(normalize_status)
    df_temp["channel_standardized"] = df_temp["ticket_channel"].apply(normalize_channel)
    df_temp["priority_standardized"] = df_temp["ticket_priority"].apply(normalize_priority)

    df_temp["has_resolution"] = df_temp["resolution"].str.len() > 0

    df_temp["support_context"] = (
        "Subject: " + df_temp["ticket_subject"] + "\n"
        "Description: " + df_temp["ticket_description"] + "\n"
        "Product: " + df_temp["product_purchased"] + "\n"
        "Resolution: " + df_temp["resolution"]
    )

    return df_temp


new_batch_silver = prepare_silver_dataframe(new_batch_raw)

new_batch_silver[
    [
        "ticket_id",
        "ticket_subject",
        "status_standardized",
        "priority_standardized",
        "has_resolution"
    ]
]

,ticket_id,ticket_subject,status_standardized,priority_standardized,has_resolution
0,1,Updated product setup issue,resolved,high,True
1,NEW-999999,Refund request for damaged product,open,medium,True


In [19]:
MERGED_SILVER_FILE = SILVER_PATH / "support_tickets_silver_merged.parquet"


def module2_merge_upsert(existing_df: pd.DataFrame, updates_df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    MODULE 2E — MERGE / UPSERT

    Simulates Delta Lake MERGE behavior:
    - If ticket_id exists, update the existing record.
    - If ticket_id does not exist, insert it as a new record.
    """

    logger.info("[MODULE 2E] MERGE / UPSERT started")

    existing = existing_df.copy()
    updates = updates_df.copy()

    existing = existing.set_index("ticket_id")
    updates = updates.set_index("ticket_id")

    updated_keys = existing.index.intersection(updates.index)
    inserted_keys = updates.index.difference(existing.index)

    # Update matching ticket IDs
    existing.update(updates)

    # Insert new ticket IDs
    merged = pd.concat([
        existing,
        updates.loc[inserted_keys]
    ])

    merged = merged.reset_index()

    merged.to_parquet(MERGED_SILVER_FILE, index=False)

    merge_summary = {
        "operation": "MERGE / UPSERT",
        "updated_records": len(updated_keys),
        "inserted_records": len(inserted_keys),
        "records_before_merge": len(existing_df),
        "records_after_merge": len(merged),
        "output_path": str(MERGED_SILVER_FILE),
        "executed_at": datetime.now(UTC).isoformat()
    }

    logger.info(
        f"[MODULE 2E] MERGE completed: "
        f"updated={len(updated_keys)}, inserted={len(inserted_keys)}, total={len(merged)}"
    )

    return merged, merge_summary


merged_silver_df, merge_summary = module2_merge_upsert(
    silver_df,
    new_batch_silver
)

merge_summary

{'operation': 'MERGE / UPSERT',
 'updated_records': 1,
 'inserted_records': 1,
 'records_before_merge': 50,
 'records_after_merge': 51,
 'output_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\silver\\support_tickets_silver_merged.parquet',
 'executed_at': '2026-07-04T10:39:34.941969+00:00'}

In [20]:
affected_ticket_ids = [
    str(silver_df.iloc[0]["ticket_id"]),
    "NEW-999999"
]

merged_silver_df[
    merged_silver_df["ticket_id"].isin(affected_ticket_ids)
][
    [
        "ticket_id",
        "ticket_subject",
        "status_standardized",
        "priority_standardized",
        "resolution"
    ]
]

,ticket_id,ticket_subject,status_standardized,priority_standardized,resolution
0,1,Updated product setup issue,resolved,high,Agent provided updated setup instructions and ...
50,NEW-999999,Refund request for damaged product,open,medium,Refund request was created and routed to the b...


In [21]:
MERGED_GOLD_FILE = GOLD_PATH / "support_knowledge_base_gold_merged.parquet"


def module2_gold_from_merged_silver(silver_input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuilds Gold Knowledge Base after MERGE operation.
    """

    logger.info("[MODULE 2F] Rebuilding Gold after MERGE started")

    gold_df = silver_input_df[
        (silver_input_df["has_resolution"] == True)
        & (silver_input_df["ticket_subject"].str.len() > 0)
        & (silver_input_df["ticket_description"].str.len() > 0)
    ].copy()

    gold_df["document_id"] = "KB-" + gold_df["ticket_id"].astype(str)

    gold_df["knowledge_document"] = (
        "Customer Issue:\n"
        + gold_df["ticket_subject"]
        + "\n\nDetails:\n"
        + gold_df["ticket_description"]
        + "\n\nProduct:\n"
        + gold_df["product_purchased"]
        + "\n\nRecommended Resolution:\n"
        + gold_df["resolution"]
    )

    gold_columns = [
        "document_id",
        "ticket_id",
        "ticket_type",
        "priority_standardized",
        "channel_standardized",
        "product_purchased",
        "knowledge_document",
    ]

    gold_df = gold_df[gold_columns]

    gold_df.to_parquet(MERGED_GOLD_FILE, index=False)

    logger.info(f"[MODULE 2F] Gold after MERGE saved: {len(gold_df)} documents")

    return gold_df


gold_merged_df = module2_gold_from_merged_silver(merged_silver_df)

print("Gold before merge:", len(gold_df))
print("Gold after merge:", len(gold_merged_df))

gold_merged_df.head()

Gold before merge: 18
Gold after merge: 20


,document_id,ticket_id,ticket_type,priority_standardized,channel_standardized,product_purchased,knowledge_document
0,KB-1,1,Technical issue,high,email,GoPro Hero,Customer Issue:\nUpdated product setup issue\n...
2,KB-3,3,Technical issue,low,social_media,Dell XPS,Customer Issue:\nNetwork problem\n\nDetails:\n...
3,KB-4,4,Billing inquiry,low,social_media,Microsoft Office,Customer Issue:\nAccount access\n\nDetails:\nI...
4,KB-5,5,Billing inquiry,low,email,Autodesk AutoCAD,Customer Issue:\nData loss\n\nDetails:\nI'm ha...
10,KB-11,11,Cancellation request,high,phone,Nintendo Switch,Customer Issue:\nData loss\n\nDetails:\nI'm ha...


In [22]:
module2_final_summary = {
    "deliverable": "Deliverable 2",
    "module": "Delta Lakehouse Simulation",
    "description": "Bronze/Silver/Gold zones with schema enforcement and MERGE-like upsert logic.",
    "bronze_records": len(bronze_df),
    "silver_records_before_merge": len(silver_df),
    "silver_records_after_merge": len(merged_silver_df),
    "gold_documents_before_merge": len(gold_df),
    "gold_documents_after_merge": len(gold_merged_df),
    "schema_enforcement": schema_enforcement_result["status"],
    "merge_operation": merge_summary,
    "bronze_path": str(BRONZE_FILE),
    "silver_path": str(MERGED_SILVER_FILE),
    "gold_path": str(MERGED_GOLD_FILE),
}

module2_final_summary

{'deliverable': 'Deliverable 2',
 'module': 'Delta Lakehouse Simulation',
 'description': 'Bronze/Silver/Gold zones with schema enforcement and MERGE-like upsert logic.',
 'bronze_records': 50,
 'silver_records_before_merge': 50,
 'silver_records_after_merge': 51,
 'gold_documents_before_merge': 18,
 'gold_documents_after_merge': 20,
 'schema_enforcement': 'passed',
 'merge_operation': {'operation': 'MERGE / UPSERT',
  'updated_records': 1,
  'inserted_records': 1,
  'records_before_merge': 50,
  'records_after_merge': 51,
  'output_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\silver\\support_tickets_silver_merged.parquet',
  'executed_at': '2026-07-04T10:39:34.941969+00:00'},
 'bronze_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\bronze\\support_tickets_bronze.parquet',
 'silver_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\silver\\support_tickets_sil

### Deliverable 2 Summary

This deliverable implements a lightweight Delta Lakehouse simulation.

The platform uses three zones:

- Bronze: stores raw validated support ticket events with Kafka-style metadata.
- Silver: stores cleaned and standardized support ticket records.
- Gold: stores a curated support knowledge base for RAG.

Because Apache Spark Delta Lake could not run in the local Windows environment due to Java gateway configuration, this project implements a Delta-like simulation using Parquet files.

The implementation still demonstrates the required Lakehouse concepts:

- Bronze / Silver / Gold zones
- Schema enforcement before downstream use
- MERGE / UPSERT behavior
- Curated Gold table for AI and RAG workloads

# Deliverable 3 — RAG Pipeline
## Chunking + embeddings + vector index + hybrid search + reranking

This module builds a RAG pipeline over the Gold support knowledge base.

**What it demonstrates:**

- Chunking Gold knowledge documents.
- Dense embeddings using Sentence Transformers.
- Vector search using cosine similarity.
- BM25 keyword retrieval.
- Hybrid search using Reciprocal Rank Fusion.
- Cross-encoder reranking.

In [23]:
# MODULE 3 — RAG Pipeline Imports

import re
import numpy as np

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

In [24]:
# MODULE 3 — RAG Output Paths

RAG_PATH = BASE_DIR / "outputs" / "rag"
RAG_PATH.mkdir(parents=True, exist_ok=True)

CHUNKS_FILE = RAG_PATH / "rag_chunks.parquet"
RAG_RESULTS_FILE = RAG_PATH / "rag_search_results.json"

print("RAG output folder is ready.")

RAG output folder is ready.


In [25]:
# MODULE 3A — Load Gold Knowledge Base

gold_for_rag_df = pd.read_parquet(MERGED_GOLD_FILE)

print("Gold documents loaded for RAG:", len(gold_for_rag_df))
gold_for_rag_df.head()

Gold documents loaded for RAG: 20


,document_id,ticket_id,ticket_type,priority_standardized,channel_standardized,product_purchased,knowledge_document
0,KB-1,1,Technical issue,high,email,GoPro Hero,Customer Issue:\nUpdated product setup issue\n...
1,KB-3,3,Technical issue,low,social_media,Dell XPS,Customer Issue:\nNetwork problem\n\nDetails:\n...
2,KB-4,4,Billing inquiry,low,social_media,Microsoft Office,Customer Issue:\nAccount access\n\nDetails:\nI...
3,KB-5,5,Billing inquiry,low,email,Autodesk AutoCAD,Customer Issue:\nData loss\n\nDetails:\nI'm ha...
4,KB-11,11,Cancellation request,high,phone,Nintendo Switch,Customer Issue:\nData loss\n\nDetails:\nI'm ha...


In [26]:
# MODULE 3B — Chunking

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Splits a document into overlapping chunks.

    This simulates production RAG chunking:
    - chunk_size controls max characters per chunk
    - overlap preserves context between chunks
    """

    text = str(text).strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def module3_create_chunks(gold_df: pd.DataFrame) -> pd.DataFrame:
    """
    MODULE 3B — CHUNKING

    Creates text chunks from Gold knowledge documents.
    """

    logger.info("[MODULE 3B] Chunking started")

    chunk_rows = []

    for _, row in gold_df.iterrows():
        chunks = chunk_text(row["knowledge_document"], chunk_size=500, overlap=100)

        for i, chunk in enumerate(chunks):
            chunk_rows.append({
                "chunk_id": f"{row['document_id']}_chunk_{i}",
                "document_id": row["document_id"],
                "ticket_id": row["ticket_id"],
                "chunk_index": i,
                "chunk_text": chunk,
                "ticket_type": row["ticket_type"],
                "product_purchased": row["product_purchased"],
            })

    chunks_df = pd.DataFrame(chunk_rows)

    chunks_df.to_parquet(CHUNKS_FILE, index=False)

    logger.info(f"[MODULE 3B] Created {len(chunks_df)} chunks")

    return chunks_df


chunks_df = module3_create_chunks(gold_for_rag_df)

print("Chunks created:", len(chunks_df))
chunks_df.head()

Chunks created: 36


,chunk_id,document_id,ticket_id,chunk_index,chunk_text,ticket_type,product_purchased
0,KB-1_chunk_0,KB-1,1,0,Customer Issue:\nUpdated product setup issue\n...,Technical issue,GoPro Hero
1,KB-3_chunk_0,KB-3,3,0,Customer Issue:\nNetwork problem\n\nDetails:\n...,Technical issue,Dell XPS
2,KB-3_chunk_1,KB-3,3,1,ollow.,Technical issue,Dell XPS
3,KB-4_chunk_0,KB-4,4,0,Customer Issue:\nAccount access\n\nDetails:\nI...,Billing inquiry,Microsoft Office
4,KB-4_chunk_1,KB-4,4,1,.,Billing inquiry,Microsoft Office


In [27]:
# MODULE 3C — Embedding Generation

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def module3_generate_embeddings(chunks_df: pd.DataFrame) -> np.ndarray:
    """
    MODULE 3C — EMBEDDINGS

    Converts text chunks into dense vector embeddings.
    """

    logger.info("[MODULE 3C] Embedding generation started")

    texts = chunks_df["chunk_text"].tolist()

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    logger.info(f"[MODULE 3C] Generated embeddings shape: {embeddings.shape}")

    return embeddings


chunk_embeddings = module3_generate_embeddings(chunks_df)

chunk_embeddings.shape

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.84it/s]


(36, 384)

In [28]:
# MODULE 3D — Vector Index

class SimpleVectorIndex:
    """
    Lightweight vector index using cosine similarity.
    This simulates a vector database search layer.
    """

    def __init__(self, chunks_df: pd.DataFrame, embeddings: np.ndarray):
        self.chunks_df = chunks_df.reset_index(drop=True)
        self.embeddings = embeddings

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        query_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        scores = cosine_similarity(query_embedding, self.embeddings)[0]

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for rank, idx in enumerate(top_indices, start=1):
            row = self.chunks_df.iloc[idx].to_dict()
            row["vector_score"] = float(scores[idx])
            row["vector_rank"] = rank
            results.append(row)

        return results


vector_index = SimpleVectorIndex(chunks_df, chunk_embeddings)

print("Vector index created successfully.")

Vector index created successfully.


In [29]:
# MODULE 3E — BM25 Keyword Index

def tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", str(text).lower())


class BM25Index:
    """
    Keyword-based search index using BM25.
    """

    def __init__(self, chunks_df: pd.DataFrame):
        self.chunks_df = chunks_df.reset_index(drop=True)
        self.tokenized_corpus = [
            tokenize(text) for text in self.chunks_df["chunk_text"].tolist()
        ]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        tokenized_query = tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for rank, idx in enumerate(top_indices, start=1):
            row = self.chunks_df.iloc[idx].to_dict()
            row["bm25_score"] = float(scores[idx])
            row["bm25_rank"] = rank
            results.append(row)

        return results


bm25_index = BM25Index(chunks_df)

print("BM25 index created successfully.")

BM25 index created successfully.


In [30]:
# MODULE 3F — Hybrid Search with RRF

def reciprocal_rank_fusion(vector_results: list[dict], bm25_results: list[dict], k: int = 60, top_k: int = 5) -> list[dict]:
    """
    Combines vector search and BM25 search using Reciprocal Rank Fusion.
    """

    scores = {}
    records = {}

    for result in vector_results:
        chunk_id = result["chunk_id"]
        rank = result["vector_rank"]

        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)
        records[chunk_id] = result

    for result in bm25_results:
        chunk_id = result["chunk_id"]
        rank = result["bm25_rank"]

        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)

        if chunk_id in records:
            records[chunk_id].update(result)
        else:
            records[chunk_id] = result

    ranked_chunk_ids = sorted(scores, key=scores.get, reverse=True)

    fused_results = []

    for final_rank, chunk_id in enumerate(ranked_chunk_ids[:top_k], start=1):
        record = records[chunk_id]
        record["rrf_score"] = float(scores[chunk_id])
        record["hybrid_rank"] = final_rank
        fused_results.append(record)

    return fused_results


print("RRF fusion function ready.")

RRF fusion function ready.


In [31]:
# MODULE 3G — Cross-Encoder Reranking

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_results(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Reranks hybrid search results using a cross-encoder model.
    """

    if not candidates:
        return []

    pairs = [(query, candidate["chunk_text"]) for candidate in candidates]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        updated = candidate.copy()
        updated["rerank_score"] = float(score)
        reranked.append(updated)

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for rank, item in enumerate(reranked[:top_k], start=1):
        item["rerank_rank"] = rank

    return reranked[:top_k]


print("Cross-encoder reranker loaded successfully.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4944.45it/s]


Cross-encoder reranker loaded successfully.


In [32]:
# MODULE 3H — RAG Search Pipeline

def module3_rag_search(query: str, top_k: int = 3) -> dict:
    """
    MODULE 3 — RAG PIPELINE

    Runs:
    - Vector search
    - BM25 keyword search
    - RRF hybrid fusion
    - Cross-encoder reranking
    """

    logger.info(f"[MODULE 3] RAG search started for query: {query}")

    vector_results = vector_index.search(query, top_k=5)
    bm25_results = bm25_index.search(query, top_k=5)

    hybrid_results = reciprocal_rank_fusion(
        vector_results,
        bm25_results,
        top_k=5
    )

    final_results = rerank_results(
        query,
        hybrid_results,
        top_k=top_k
    )

    answer_context = "\n\n---\n\n".join(
        [result["chunk_text"] for result in final_results]
    )

    rag_output = {
        "query": query,
        "retrieval_pipeline": [
            "vector_search",
            "bm25_keyword_search",
            "rrf_hybrid_fusion",
            "cross_encoder_reranking"
        ],
        "top_results": final_results,
        "answer_context": answer_context,
        "generated_at": datetime.now(UTC).isoformat()
    }

    with open(RAG_RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(rag_output, f, indent=2, ensure_ascii=False, default=str)

    logger.info("[MODULE 3] RAG search completed")

    return rag_output

In [33]:
test_query = "How can I solve a product setup issue?"

rag_result = module3_rag_search(test_query, top_k=3)

print("Query:", rag_result["query"])
print("\nTop Retrieved Context:\n")
print(rag_result["answer_context"][:1500])

Query: How can I solve a product setup issue?

Top Retrieved Context:

Customer Issue:
Updated product setup issue

Details:
Customer provided additional details about the setup issue.

Product:
GoPro Hero

Recommended Resolution:
Agent provided updated setup instructions and confirmed the issue was resolved.

---

Customer Issue:
Product setup

Details:
I'm having an issue with the {product_purchased}. Please assist.

Product Name: TPUBASK3E3KQ0


Join Date: Oct 2007 Posts: 11,532

Quote: I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?

Product:
Sony PlayStation

Recommended Resolution:
Officer moment world sing parent available.

---

Customer Issue:
Product compatibility

Details:
I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.

If I'd just switched to a The issue I'm facing is interm

In [34]:
rag_results_table = pd.DataFrame([
    {
        "rerank_rank": item.get("rerank_rank"),
        "chunk_id": item.get("chunk_id"),
        "ticket_id": item.get("ticket_id"),
        "vector_score": item.get("vector_score"),
        "bm25_score": item.get("bm25_score"),
        "rrf_score": item.get("rrf_score"),
        "rerank_score": item.get("rerank_score"),
        "preview": item.get("chunk_text", "")[:120]
    }
    for item in rag_result["top_results"]
])

rag_results_table

,rerank_rank,chunk_id,ticket_id,vector_score,bm25_score,rrf_score,rerank_score,preview
0,1,KB-1_chunk_0,1,0.482800,6.850092,0.032266,1.485818,Customer Issue:\nUpdated product setup issue\n...
1,2,KB-15_chunk_0,15,0.484217,4.036343,0.032258,0.719211,Customer Issue:\nProduct setup\n\nDetails:\nI'...
2,3,KB-42_chunk_0,42,0.499051,NaN,0.016393,-4.182296,Customer Issue:\nProduct compatibility\n\nDeta...


In [35]:
module3_summary = {
    "deliverable": "Deliverable 3",
    "module": "RAG Pipeline",
    "gold_documents": len(gold_for_rag_df),
    "chunks_created": len(chunks_df),
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_index": "SimpleVectorIndex using cosine similarity",
    "keyword_index": "BM25Okapi",
    "hybrid_search": "Reciprocal Rank Fusion",
    "reranking_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "test_query": test_query,
    "top_results": len(rag_result["top_results"]),
    "rag_results_path": str(RAG_RESULTS_FILE),
}

module3_summary

{'deliverable': 'Deliverable 3',
 'module': 'RAG Pipeline',
 'gold_documents': 20,
 'chunks_created': 36,
 'embedding_model': 'all-MiniLM-L6-v2',
 'vector_index': 'SimpleVectorIndex using cosine similarity',
 'keyword_index': 'BM25Okapi',
 'hybrid_search': 'Reciprocal Rank Fusion',
 'reranking_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
 'test_query': 'How can I solve a product setup issue?',
 'top_results': 3,
 'rag_results_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\rag\\rag_search_results.json'}

### Deliverable 3 Summary

This deliverable implements an advanced RAG pipeline over the Gold support knowledge base.

The pipeline includes:

- Chunking support knowledge documents into overlapping text chunks.
- Generating dense embeddings using Sentence Transformers.
- Building a lightweight vector index using cosine similarity.
- Building a BM25 keyword index.
- Combining vector and keyword retrieval using Reciprocal Rank Fusion.
- Applying cross-encoder reranking for higher precision retrieval.

The final output is a ranked set of support knowledge chunks that can be used to generate customer support answer suggestions.

# Deliverable 4 — Orchestration
## Airflow-style DAG connecting all modules end-to-end

This module connects the pipeline steps into a DAG-style workflow:

1. Ingestion
2. Lakehouse storage
3. RAG pipeline

A lightweight Airflow-style runner is used inside the notebook, and a reference Airflow DAG file is generated under `dags/`.

In [36]:
# MODULE 4 — Orchestration Output Paths

ORCHESTRATION_PATH = BASE_DIR / "outputs" / "orchestration"
ORCHESTRATION_PATH.mkdir(parents=True, exist_ok=True)

DAG_RUN_REPORT_FILE = ORCHESTRATION_PATH / "dag_run_report.json"

print("Orchestration output folder is ready.")

Orchestration output folder is ready.


In [37]:
# MODULE 4A — Airflow-style DAG Engine

class AirflowStyleDAG:
    """
    A lightweight Airflow-style DAG simulator.

    It stores tasks, dependencies, and executes them in order.
    This is used because running a full Airflow server is heavy
    for a two-day local JupyterLab capstone.
    """

    def __init__(self, dag_id: str):
        self.dag_id = dag_id
        self.tasks = {}
        self.dependencies = defaultdict(list)
        self.execution_log = []

    def task(self, task_id: str):
        def decorator(func):
            self.tasks[task_id] = func
            return func
        return decorator

    def set_dependency(self, upstream_task_id: str, downstream_task_id: str):
        self.dependencies[downstream_task_id].append(upstream_task_id)

    def _can_run(self, task_id: str, completed_tasks: set) -> bool:
        upstream_tasks = self.dependencies.get(task_id, [])
        return all(task in completed_tasks for task in upstream_tasks)

    def run(self) -> dict:
        logger.info(f"[MODULE 4] DAG started: {self.dag_id}")

        completed_tasks = set()
        context = {}
        run_id = str(uuid.uuid4())

        while len(completed_tasks) < len(self.tasks):
            progress_made = False

            for task_id, task_func in self.tasks.items():
                if task_id in completed_tasks:
                    continue

                if self._can_run(task_id, completed_tasks):
                    logger.info(f"[MODULE 4] Running task: {task_id}")

                    start_time = datetime.now(UTC).isoformat()

                    try:
                        output = task_func(context)

                        end_time = datetime.now(UTC).isoformat()

                        self.execution_log.append({
                            "task_id": task_id,
                            "status": "success",
                            "start_time": start_time,
                            "end_time": end_time
                        })

                        context[task_id] = output
                        completed_tasks.add(task_id)
                        progress_made = True

                        logger.info(f"[MODULE 4] Task completed: {task_id}")

                    except Exception as e:
                        end_time = datetime.now(UTC).isoformat()

                        self.execution_log.append({
                            "task_id": task_id,
                            "status": "failed",
                            "start_time": start_time,
                            "end_time": end_time,
                            "error": str(e)
                        })

                        logger.error(f"[MODULE 4] Task failed: {task_id} — {e}")

                        raise

            if not progress_made:
                raise RuntimeError("DAG execution stopped because dependencies could not be resolved.")

        dag_report = {
            "deliverable": "Deliverable 4",
            "dag_id": self.dag_id,
            "run_id": run_id,
            "status": "success",
            "tasks_executed": [log["task_id"] for log in self.execution_log],
            "execution_log": self.execution_log,
            "completed_at": datetime.now(UTC).isoformat()
        }

        with open(DAG_RUN_REPORT_FILE, "w", encoding="utf-8") as f:
            json.dump(dag_report, f, indent=2, ensure_ascii=False, default=str)

        logger.info(f"[MODULE 4] DAG completed: {self.dag_id}")

        return dag_report

In [38]:
# MODULE 4B — Define DAG Tasks

customer_support_dag = AirflowStyleDAG(
    dag_id="customer_support_intelligence_platform_dag"
)


@customer_support_dag.task("task_1_ingestion")
def task_1_ingestion(context):
    kafka = MockKafka()

    produced_count = module1_ingest(
        kafka=kafka,
        df=df_raw,
        limit=50
    )

    validation_summary = module1_consume_and_validate(kafka)

    return {
        "produced_count": produced_count,
        "validation_summary": validation_summary
    }


@customer_support_dag.task("task_2_lakehouse_storage")
def task_2_lakehouse_storage(context):
    bronze_df_local = module2_bronze_load(VALID_EVENTS_FILE)
    silver_df_local = module2_silver_transform(bronze_df_local)
    gold_df_local = module2_gold_create_knowledge_base(silver_df_local)

    schema_result = module2_schema_enforcement(
        silver_df_local,
        EXPECTED_SILVER_COLUMNS
    )

    # Create a small batch for MERGE
    new_batch_local = pd.DataFrame([
        {
            "ticket_id": str(silver_df_local.iloc[0]["ticket_id"]),
            "ticket_subject": "Updated product setup issue",
            "ticket_description": "Customer provided additional details about the setup issue.",
            "ticket_type": "Technical issue",
            "ticket_status": "Closed",
            "resolution": "Agent provided updated setup instructions and confirmed the issue was resolved.",
            "ticket_priority": "High",
            "ticket_channel": "Email",
            "product_purchased": silver_df_local.iloc[0]["product_purchased"],
            "date_of_purchase": silver_df_local.iloc[0]["date_of_purchase"],
            "kafka_topic": RAW_TOPIC,
            "kafka_offset": 9999,
            "event_time": datetime.now(UTC).isoformat(),
            "ingestion_time": datetime.now(UTC).isoformat(),
        },
        {
            "ticket_id": "NEW-999999",
            "ticket_subject": "Refund request for damaged product",
            "ticket_description": "Customer received a damaged product and requests a refund.",
            "ticket_type": "Refund request",
            "ticket_status": "Open",
            "resolution": "Refund request was created and routed to the billing team.",
            "ticket_priority": "Medium",
            "ticket_channel": "Chat",
            "product_purchased": "Wireless Headphones",
            "date_of_purchase": "2024-01-15",
            "kafka_topic": RAW_TOPIC,
            "kafka_offset": 10000,
            "event_time": datetime.now(UTC).isoformat(),
            "ingestion_time": datetime.now(UTC).isoformat(),
        }
    ])

    new_batch_silver_local = prepare_silver_dataframe(new_batch_local)

    merged_silver_df_local, merge_summary_local = module2_merge_upsert(
        silver_df_local,
        new_batch_silver_local
    )

    gold_merged_df_local = module2_gold_from_merged_silver(
        merged_silver_df_local
    )

    return {
        "bronze_records": len(bronze_df_local),
        "silver_records_before_merge": len(silver_df_local),
        "silver_records_after_merge": len(merged_silver_df_local),
        "gold_documents_before_merge": len(gold_df_local),
        "gold_documents_after_merge": len(gold_merged_df_local),
        "schema_enforcement": schema_result,
        "merge_summary": merge_summary_local,
        "gold_path": str(MERGED_GOLD_FILE)
    }


@customer_support_dag.task("task_3_rag_pipeline")
def task_3_rag_pipeline(context):
    global gold_for_rag_df
    global chunks_df
    global chunk_embeddings
    global vector_index
    global bm25_index

    gold_for_rag_df = pd.read_parquet(MERGED_GOLD_FILE)

    chunks_df = module3_create_chunks(gold_for_rag_df)
    chunk_embeddings = module3_generate_embeddings(chunks_df)

    vector_index = SimpleVectorIndex(chunks_df, chunk_embeddings)
    bm25_index = BM25Index(chunks_df)

    query = "How can I solve a product setup issue?"

    rag_output = module3_rag_search(
        query=query,
        top_k=3
    )

    return {
        "gold_documents": len(gold_for_rag_df),
        "chunks_created": len(chunks_df),
        "test_query": query,
        "top_results": len(rag_output["top_results"]),
        "rag_results_path": str(RAG_RESULTS_FILE)
    }

In [39]:
# MODULE 4C — Define DAG Dependencies

customer_support_dag.set_dependency(
    "task_1_ingestion",
    "task_2_lakehouse_storage"
)

customer_support_dag.set_dependency(
    "task_2_lakehouse_storage",
    "task_3_rag_pipeline"
)

print("DAG dependencies are ready.")

DAG dependencies are ready.


In [40]:
dag_report = customer_support_dag.run()

dag_report

Batches: 100%|██████████| 2/2 [00:00<00:00,  4.45it/s]


{'deliverable': 'Deliverable 4',
 'dag_id': 'customer_support_intelligence_platform_dag',
 'run_id': '01d7b89c-9203-4dac-a15c-e3c74a6298d6',
 'status': 'success',
 'tasks_executed': ['task_1_ingestion',
  'task_2_lakehouse_storage',
  'task_3_rag_pipeline'],
 'execution_log': [{'task_id': 'task_1_ingestion',
   'status': 'success',
   'start_time': '2026-07-04T10:40:02.543638+00:00',
   'end_time': '2026-07-04T10:40:02.581080+00:00'},
  {'task_id': 'task_2_lakehouse_storage',
   'status': 'success',
   'start_time': '2026-07-04T10:40:02.581256+00:00',
   'end_time': '2026-07-04T10:40:02.647973+00:00'},
  {'task_id': 'task_3_rag_pipeline',
   'status': 'success',
   'start_time': '2026-07-04T10:40:02.648366+00:00',
   'end_time': '2026-07-04T10:40:03.219760+00:00'}],
 'completed_at': '2026-07-04T10:40:03.219851+00:00'}

In [41]:
pd.DataFrame(dag_report["execution_log"])

,task_id,status,start_time,end_time
0,task_1_ingestion,success,2026-07-04T10:40:02.543638+00:00,2026-07-04T10:40:02.581080+00:00
1,task_2_lakehouse_storage,success,2026-07-04T10:40:02.581256+00:00,2026-07-04T10:40:02.647973+00:00
2,task_3_rag_pipeline,success,2026-07-04T10:40:02.648366+00:00,2026-07-04T10:40:03.219760+00:00


In [42]:
# MODULE 4D — Create a reference Airflow DAG file for GitHub

DAGS_PATH = BASE_DIR / "dags"
DAGS_PATH.mkdir(parents=True, exist_ok=True)

AIRFLOW_DAG_FILE = DAGS_PATH / "customer_support_intelligence_dag.py"

airflow_dag_code = '''
"""
Reference Airflow DAG for the Customer Support Intelligence Platform.

This file shows how the pipeline would be orchestrated in Apache Airflow.
In this capstone, the actual execution is simulated inside the Jupyter notebook
using AirflowStyleDAG to keep the project lightweight and runnable locally.
"""

from datetime import datetime

try:
    from airflow import DAG
    from airflow.operators.python import PythonOperator
except ImportError:
    DAG = None
    PythonOperator = None


def run_ingestion():
    print("Run ingestion: Kafka producer + consumer + schema validation")


def run_lakehouse_storage():
    print("Run Bronze/Silver/Gold lakehouse + schema enforcement + MERGE")


def run_rag_pipeline():
    print("Run RAG pipeline: chunking + embeddings + hybrid search + reranking")


if DAG is not None:
    with DAG(
        dag_id="customer_support_intelligence_platform",
        start_date=datetime(2026, 1, 1),
        schedule_interval=None,
        catchup=False,
        tags=["capstone", "data-engineering", "rag"],
    ) as dag:

        ingestion = PythonOperator(
            task_id="ingestion",
            python_callable=run_ingestion,
        )

        lakehouse_storage = PythonOperator(
            task_id="lakehouse_storage",
            python_callable=run_lakehouse_storage,
        )

        rag_pipeline = PythonOperator(
            task_id="rag_pipeline",
            python_callable=run_rag_pipeline,
        )

        ingestion >> lakehouse_storage >> rag_pipeline
'''

with open(AIRFLOW_DAG_FILE, "w", encoding="utf-8") as f:
    f.write(airflow_dag_code)

print("Airflow DAG reference file created:")
print(AIRFLOW_DAG_FILE)

Airflow DAG reference file created:
C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\dags\customer_support_intelligence_dag.py


In [43]:
module4_summary = {
    "deliverable": "Deliverable 4",
    "module": "Orchestration",
    "orchestration_type": "Airflow-style DAG simulation",
    "dag_id": dag_report["dag_id"],
    "dag_status": dag_report["status"],
    "tasks_count": len(dag_report["tasks_executed"]),
    "tasks_executed": dag_report["tasks_executed"],
    "dag_run_report_path": str(DAG_RUN_REPORT_FILE),
    "airflow_reference_dag_file": str(AIRFLOW_DAG_FILE),
}

module4_summary

{'deliverable': 'Deliverable 4',
 'module': 'Orchestration',
 'orchestration_type': 'Airflow-style DAG simulation',
 'dag_id': 'customer_support_intelligence_platform_dag',
 'dag_status': 'success',
 'tasks_count': 3,
 'tasks_executed': ['task_1_ingestion',
  'task_2_lakehouse_storage',
  'task_3_rag_pipeline'],
 'dag_run_report_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\orchestration\\dag_run_report.json',
 'airflow_reference_dag_file': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\dags\\customer_support_intelligence_dag.py'}

### Deliverable 4 Summary

This deliverable implements an Airflow-style orchestration layer.

The DAG connects the main pipeline modules end-to-end:

1. Ingestion: Mock Kafka producer, consumer, and schema validation.
2. Lakehouse Storage: Bronze, Silver, and Gold layers with schema enforcement and MERGE / UPSERT.
3. RAG Pipeline: Chunking, embeddings, vector search, BM25 keyword search, hybrid fusion, and reranking.

A lightweight Airflow-style DAG runner is used inside the notebook to keep the project runnable in a local JupyterLab environment.  
A reference Airflow DAG file is also generated under the `dags/` folder for GitHub submission.

# Deliverable 5 — Quality Gate and OpenLineage Events
## Great Expectations-style suite + OpenLineage events emitted

This module implements the quality and lineage layer.

**What it demonstrates:**

- Great Expectations-style validation suite.
- Quality report generation.
- Failed-quality records output.
- OpenLineage-style JSONL event emission for the full pipeline.

In [44]:
# MODULE 5 — Quality Gate and OpenLineage Paths

QUALITY_PATH = BASE_DIR / "outputs" / "quality"
LINEAGE_PATH = BASE_DIR / "outputs" / "lineage"

QUALITY_PATH.mkdir(parents=True, exist_ok=True)
LINEAGE_PATH.mkdir(parents=True, exist_ok=True)

GE_SUITE_FILE = QUALITY_PATH / "great_expectations_suite.json"
QUALITY_REPORT_FILE = QUALITY_PATH / "quality_report.json"
FAILED_QUALITY_FILE = QUALITY_PATH / "failed_quality_records.parquet"

OPENLINEAGE_EVENTS_FILE = LINEAGE_PATH / "openlineage_events.jsonl"

print("Quality and lineage output folders are ready.")

Quality and lineage output folders are ready.


In [45]:
# MODULE 5A — Load Final Tables for Quality Gate

silver_for_quality_df = pd.read_parquet(MERGED_SILVER_FILE)
gold_for_quality_df = pd.read_parquet(MERGED_GOLD_FILE)

print("Silver records:", len(silver_for_quality_df))
print("Gold records:", len(gold_for_quality_df))

Silver records: 51
Gold records: 20


In [46]:
# MODULE 5B — Great Expectations-style Suite

def build_great_expectations_suite() -> dict:
    """
    Builds a lightweight Great Expectations-style suite.

    This mirrors common Great Expectations checks using expectation names,
    while keeping the project runnable in local JupyterLab without a full GX setup.
    """

    suite = {
        "suite_name": "customer_support_quality_suite",
        "framework": "Great Expectations-style validation suite",
        "expectations": [
            {
                "expectation_type": "expect_column_to_exist",
                "column": "ticket_id",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_column_values_to_not_be_null",
                "column": "ticket_id",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_column_values_to_be_unique",
                "column": "ticket_id",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_column_values_to_not_be_empty",
                "column": "ticket_subject",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_column_values_to_not_be_empty",
                "column": "ticket_description",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_column_values_to_be_in_set",
                "column": "priority_standardized",
                "value_set": ["low", "medium", "high", "critical", "unknown"],
                "severity": "warning"
            },
            {
                "expectation_type": "expect_column_values_to_be_in_set",
                "column": "channel_standardized",
                "value_set": ["email", "chat", "phone", "social_media", "other"],
                "severity": "warning"
            },
            {
                "expectation_type": "expect_gold_documents_to_have_content",
                "column": "knowledge_document",
                "severity": "critical"
            },
            {
                "expectation_type": "expect_gold_layer_to_exclude_pii_columns",
                "columns_not_allowed": ["customer_name", "customer_email"],
                "severity": "critical"
            }
        ],
        "created_at": datetime.now(UTC).isoformat()
    }

    with open(GE_SUITE_FILE, "w", encoding="utf-8") as f:
        json.dump(suite, f, indent=2, ensure_ascii=False)

    return suite


ge_suite = build_great_expectations_suite()

ge_suite

{'suite_name': 'customer_support_quality_suite',
 'framework': 'Great Expectations-style validation suite',
 'expectations': [{'expectation_type': 'expect_column_to_exist',
   'column': 'ticket_id',
   'severity': 'critical'},
  {'expectation_type': 'expect_column_values_to_not_be_null',
   'column': 'ticket_id',
   'severity': 'critical'},
  {'expectation_type': 'expect_column_values_to_be_unique',
   'column': 'ticket_id',
   'severity': 'critical'},
  {'expectation_type': 'expect_column_values_to_not_be_empty',
   'column': 'ticket_subject',
   'severity': 'critical'},
  {'expectation_type': 'expect_column_values_to_not_be_empty',
   'column': 'ticket_description',
   'severity': 'critical'},
  {'expectation_type': 'expect_column_values_to_be_in_set',
   'column': 'priority_standardized',
   'value_set': ['low', 'medium', 'high', 'critical', 'unknown'],
   'severity': 'warning'},
  {'expectation_type': 'expect_column_values_to_be_in_set',
   'column': 'channel_standardized',
   'val

In [47]:
# MODULE 5C — Run Quality Gate

def run_expectation(expectation: dict, silver_df: pd.DataFrame, gold_df: pd.DataFrame) -> dict:
    expectation_type = expectation["expectation_type"]
    severity = expectation.get("severity", "warning")

    passed = True
    details = {}

    if expectation_type == "expect_column_to_exist":
        column = expectation["column"]
        passed = column in silver_df.columns
        details = {"column": column, "exists": passed}

    elif expectation_type == "expect_column_values_to_not_be_null":
        column = expectation["column"]
        null_count = int(silver_df[column].isna().sum())
        passed = null_count == 0
        details = {"column": column, "null_count": null_count}

    elif expectation_type == "expect_column_values_to_be_unique":
        column = expectation["column"]
        duplicate_count = int(silver_df[column].duplicated().sum())
        passed = duplicate_count == 0
        details = {"column": column, "duplicate_count": duplicate_count}

    elif expectation_type == "expect_column_values_to_not_be_empty":
        column = expectation["column"]
        empty_count = int((silver_df[column].apply(clean_text).str.len() == 0).sum())
        passed = empty_count == 0
        details = {"column": column, "empty_count": empty_count}

    elif expectation_type == "expect_column_values_to_be_in_set":
        column = expectation["column"]
        value_set = set(expectation["value_set"])
        invalid_count = int((~silver_df[column].isin(value_set)).sum())
        passed = invalid_count == 0
        details = {
            "column": column,
            "allowed_values": sorted(list(value_set)),
            "invalid_count": invalid_count
        }

    elif expectation_type == "expect_gold_documents_to_have_content":
        column = expectation["column"]
        empty_gold_docs = int((gold_df[column].apply(clean_text).str.len() == 0).sum())
        passed = empty_gold_docs == 0 and len(gold_df) > 0
        details = {
            "column": column,
            "gold_records": len(gold_df),
            "empty_documents": empty_gold_docs
        }

    elif expectation_type == "expect_gold_layer_to_exclude_pii_columns":
        columns_not_allowed = expectation["columns_not_allowed"]
        found_pii_columns = [
            col for col in columns_not_allowed
            if col in gold_df.columns
        ]
        passed = len(found_pii_columns) == 0
        details = {
            "columns_not_allowed": columns_not_allowed,
            "found_pii_columns": found_pii_columns
        }

    else:
        passed = False
        details = {"error": f"Unsupported expectation: {expectation_type}"}

    return {
        "expectation_type": expectation_type,
        "severity": severity,
        "passed": passed,
        "details": details,
        "checked_at": datetime.now(UTC).isoformat()
    }


def module5_quality_gate(silver_df: pd.DataFrame, gold_df: pd.DataFrame, suite: dict) -> dict:
    """
    MODULE 5 — QUALITY GATE

    Runs a Great Expectations-style validation suite.
    Critical failures fail the quality gate.
    """

    logger.info("[MODULE 5] Quality gate started")

    expectation_results = []

    for expectation in suite["expectations"]:
        result = run_expectation(expectation, silver_df, gold_df)
        expectation_results.append(result)

    critical_failures = [
        r for r in expectation_results
        if (not r["passed"]) and r["severity"] == "critical"
    ]

    warning_failures = [
        r for r in expectation_results
        if (not r["passed"]) and r["severity"] == "warning"
    ]

    passed_expectations = sum(1 for r in expectation_results if r["passed"])
    total_expectations = len(expectation_results)

    quality_gate_passed = len(critical_failures) == 0

    # Extra data profile
    data_profile = {
        "silver_records": len(silver_df),
        "gold_records": len(gold_df),
        "silver_missing_resolution_records": int((silver_df["resolution"].apply(clean_text).str.len() == 0).sum()),
        "gold_documents_available": len(gold_df),
        "unique_ticket_ids": int(silver_df["ticket_id"].nunique())
    }

    quality_report = {
        "deliverable": "Deliverable 5",
        "module": "Quality Gate",
        "suite_name": suite["suite_name"],
        "framework": suite["framework"],
        "quality_gate_passed": quality_gate_passed,
        "total_expectations": total_expectations,
        "passed_expectations": passed_expectations,
        "failed_expectations": total_expectations - passed_expectations,
        "critical_failures": len(critical_failures),
        "warning_failures": len(warning_failures),
        "data_profile": data_profile,
        "results": expectation_results,
        "generated_at": datetime.now(UTC).isoformat()
    }

    with open(QUALITY_REPORT_FILE, "w", encoding="utf-8") as f:
        json.dump(quality_report, f, indent=2, ensure_ascii=False, default=str)

    logger.info(
        f"[MODULE 5] Quality gate completed: passed={quality_gate_passed}"
    )

    return quality_report


quality_report = module5_quality_gate(
    silver_for_quality_df,
    gold_for_quality_df,
    ge_suite
)

quality_report

{'deliverable': 'Deliverable 5',
 'module': 'Quality Gate',
 'suite_name': 'customer_support_quality_suite',
 'framework': 'Great Expectations-style validation suite',
 'quality_gate_passed': True,
 'total_expectations': 9,
 'passed_expectations': 9,
 'failed_expectations': 0,
 'critical_failures': 0,
 'warning_failures': 0,
 'data_profile': {'silver_records': 51,
  'gold_records': 20,
  'silver_missing_resolution_records': 31,
  'gold_documents_available': 20,
  'unique_ticket_ids': 51},
 'results': [{'expectation_type': 'expect_column_to_exist',
   'severity': 'critical',
   'passed': True,
   'details': {'column': 'ticket_id', 'exists': True},
   'checked_at': '2026-07-04T10:40:03.327331+00:00'},
  {'expectation_type': 'expect_column_values_to_not_be_null',
   'severity': 'critical',
   'passed': True,
   'details': {'column': 'ticket_id', 'null_count': 0},
   'checked_at': '2026-07-04T10:40:03.327816+00:00'},
  {'expectation_type': 'expect_column_values_to_be_unique',
   'severity'

In [48]:
# MODULE 5D — Failed Quality Records

def identify_failed_quality_records(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies records with serious data quality issues.
    """

    failed_df = df[
        (df["ticket_id"].isna())
        | (df["ticket_subject"].apply(clean_text).str.len() == 0)
        | (df["ticket_description"].apply(clean_text).str.len() == 0)
        | (~df["priority_standardized"].isin({"low", "medium", "high", "critical", "unknown"}))
        | (~df["channel_standardized"].isin({"email", "chat", "phone", "social_media", "other"}))
    ].copy()

    failed_df.to_parquet(FAILED_QUALITY_FILE, index=False)

    return failed_df


failed_quality_df = identify_failed_quality_records(silver_for_quality_df)

print("Failed quality records:", len(failed_quality_df))
failed_quality_df.head()

Failed quality records: 0


,ticket_id,ticket_subject,ticket_description,ticket_type,ticket_status,resolution,ticket_priority,ticket_channel,product_purchased,date_of_purchase,kafka_topic,kafka_offset,event_time,ingestion_time,status_standardized,channel_standardized,priority_standardized,has_resolution,support_context


In [49]:
# MODULE 5E — OpenLineage Event Emitter

def write_jsonl_line(path: Path, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def emit_openlineage_event(
    event_type: str,
    job_name: str,
    run_id: str,
    inputs: list[dict],
    outputs: list[dict],
    facets: dict | None = None
) -> dict:
    """
    Emits an OpenLineage-style event as JSONL.
    """

    event = {
        "eventType": event_type,
        "eventTime": datetime.now(UTC).isoformat(),
        "producer": "customer-support-intelligence-platform",
        "schemaURL": "https://openlineage.io/spec/1-0-5/OpenLineage.json",
        "run": {
            "runId": run_id
        },
        "job": {
            "namespace": "local_jupyter_capstone",
            "name": job_name
        },
        "inputs": inputs,
        "outputs": outputs,
        "facets": facets or {}
    }

    write_jsonl_line(OPENLINEAGE_EVENTS_FILE, event)

    return event

In [50]:
# MODULE 5F — Emit OpenLineage Events for the Full Pipeline

def dataset_ref(name: str, path: str) -> dict:
    return {
        "namespace": "local_file_system",
        "name": name,
        "facets": {
            "storage": {
                "path": path
            }
        }
    }


def module5_emit_openlineage_events(
    dag_report: dict,
    quality_report: dict
) -> dict:
    """
    Emits OpenLineage-style events for each major pipeline step.
    """

    logger.info("[MODULE 5] OpenLineage event emission started")

    if OPENLINEAGE_EVENTS_FILE.exists():
        OPENLINEAGE_EVENTS_FILE.unlink()

    run_id = dag_report["run_id"]

    emitted_events = []

    jobs = [
        {
            "job_name": "module1_ingestion",
            "inputs": [
                dataset_ref("kaggle_customer_support_dataset", KAGGLE_DATASET)
            ],
            "outputs": [
                dataset_ref("valid_events_jsonl", str(VALID_EVENTS_FILE)),
                dataset_ref("invalid_events_jsonl", str(INVALID_EVENTS_FILE))
            ],
            "facets": {
                "records": {
                    "produced": 50,
                    "valid": 50,
                    "invalid": 0
                }
            }
        },
        {
            "job_name": "module2_bronze_silver_gold_lakehouse",
            "inputs": [
                dataset_ref("valid_events_jsonl", str(VALID_EVENTS_FILE))
            ],
            "outputs": [
                dataset_ref("bronze_support_tickets", str(BRONZE_FILE)),
                dataset_ref("merged_silver_support_tickets", str(MERGED_SILVER_FILE)),
                dataset_ref("merged_gold_knowledge_base", str(MERGED_GOLD_FILE))
            ],
            "facets": {
                "lakehouse": {
                    "type": "Delta Lakehouse simulation",
                    "zones": ["bronze", "silver", "gold"],
                    "schema_enforcement": "passed",
                    "merge": "updated=1, inserted=1"
                }
            }
        },
        {
            "job_name": "module3_rag_pipeline",
            "inputs": [
                dataset_ref("merged_gold_knowledge_base", str(MERGED_GOLD_FILE))
            ],
            "outputs": [
                dataset_ref("rag_chunks", str(CHUNKS_FILE)),
                dataset_ref("rag_search_results", str(RAG_RESULTS_FILE))
            ],
            "facets": {
                "rag": {
                    "chunking": True,
                    "embedding_model": "all-MiniLM-L6-v2",
                    "vector_index": "cosine_similarity",
                    "keyword_index": "BM25",
                    "hybrid_search": "RRF",
                    "reranking": "cross-encoder/ms-marco-MiniLM-L-6-v2"
                }
            }
        },
        {
            "job_name": "module4_airflow_style_orchestration",
            "inputs": [
                dataset_ref("pipeline_modules", "notebook_modules")
            ],
            "outputs": [
                dataset_ref("dag_run_report", str(DAG_RUN_REPORT_FILE)),
                dataset_ref("airflow_reference_dag", str(AIRFLOW_DAG_FILE))
            ],
            "facets": {
                "orchestration": {
                    "dag_id": dag_report["dag_id"],
                    "status": dag_report["status"],
                    "tasks_count": len(dag_report["execution_log"])
                }
            }
        },
        {
            "job_name": "module5_quality_gate",
            "inputs": [
                dataset_ref("merged_silver_support_tickets", str(MERGED_SILVER_FILE)),
                dataset_ref("merged_gold_knowledge_base", str(MERGED_GOLD_FILE))
            ],
            "outputs": [
                dataset_ref("great_expectations_suite", str(GE_SUITE_FILE)),
                dataset_ref("quality_report", str(QUALITY_REPORT_FILE)),
                dataset_ref("failed_quality_records", str(FAILED_QUALITY_FILE))
            ],
            "facets": {
                "quality": {
                    "suite_name": quality_report["suite_name"],
                    "quality_gate_passed": quality_report["quality_gate_passed"],
                    "total_expectations": quality_report["total_expectations"],
                    "critical_failures": quality_report["critical_failures"],
                    "warning_failures": quality_report["warning_failures"]
                }
            }
        }
    ]

    for job in jobs:
        emitted_events.append(
            emit_openlineage_event(
                event_type="START",
                job_name=job["job_name"],
                run_id=run_id,
                inputs=job["inputs"],
                outputs=job["outputs"],
                facets=job["facets"]
            )
        )

        emitted_events.append(
            emit_openlineage_event(
                event_type="COMPLETE",
                job_name=job["job_name"],
                run_id=run_id,
                inputs=job["inputs"],
                outputs=job["outputs"],
                facets=job["facets"]
            )
        )

    logger.info(f"[MODULE 5] OpenLineage events emitted: {len(emitted_events)}")

    return {
        "events_emitted": len(emitted_events),
        "openlineage_events_path": str(OPENLINEAGE_EVENTS_FILE),
        "run_id": run_id
    }


openlineage_summary = module5_emit_openlineage_events(
    dag_report=dag_report,
    quality_report=quality_report
)

openlineage_summary

{'events_emitted': 10,
 'openlineage_events_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\lineage\\openlineage_events.jsonl',
 'run_id': '01d7b89c-9203-4dac-a15c-e3c74a6298d6'}

In [51]:
# MODULE 5G — Preview OpenLineage Events

with open(OPENLINEAGE_EVENTS_FILE, "r", encoding="utf-8") as f:
    sample_lineage_events = [json.loads(next(f)) for _ in range(3)]

sample_lineage_events

[{'eventType': 'START',
  'eventTime': '2026-07-04T10:40:03.400534+00:00',
  'producer': 'customer-support-intelligence-platform',
  'schemaURL': 'https://openlineage.io/spec/1-0-5/OpenLineage.json',
  'run': {'runId': '01d7b89c-9203-4dac-a15c-e3c74a6298d6'},
  'job': {'namespace': 'local_jupyter_capstone', 'name': 'module1_ingestion'},
  'inputs': [{'namespace': 'local_file_system',
    'name': 'kaggle_customer_support_dataset',
    'facets': {'storage': {'path': 'suraj520/customer-support-ticket-dataset'}}}],
  'outputs': [{'namespace': 'local_file_system',
    'name': 'valid_events_jsonl',
    'facets': {'storage': {'path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\valid_events.jsonl'}}},
   {'namespace': 'local_file_system',
    'name': 'invalid_events_jsonl',
    'facets': {'storage': {'path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\invalid_events.jsonl'}}}],
  'facets': {'r

In [52]:
module5_summary = {
    "deliverable": "Deliverable 5",
    "module": "Quality Gate and OpenLineage",
    "quality_framework": "Great Expectations-style validation suite",
    "quality_gate_passed": quality_report["quality_gate_passed"],
    "total_expectations": quality_report["total_expectations"],
    "passed_expectations": quality_report["passed_expectations"],
    "failed_expectations": quality_report["failed_expectations"],
    "openlineage_events_emitted": openlineage_summary["events_emitted"],
    "great_expectations_suite_path": str(GE_SUITE_FILE),
    "quality_report_path": str(QUALITY_REPORT_FILE),
    "failed_quality_records_path": str(FAILED_QUALITY_FILE),
    "openlineage_events_path": str(OPENLINEAGE_EVENTS_FILE),
}

module5_summary

{'deliverable': 'Deliverable 5',
 'module': 'Quality Gate and OpenLineage',
 'quality_framework': 'Great Expectations-style validation suite',
 'quality_gate_passed': True,
 'total_expectations': 9,
 'passed_expectations': 9,
 'failed_expectations': 0,
 'openlineage_events_emitted': 10,
 'great_expectations_suite_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\quality\\great_expectations_suite.json',
 'quality_report_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\quality\\quality_report.json',
 'failed_quality_records_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\quality\\failed_quality_records.parquet',
 'openlineage_events_path': 'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\lineage\\openlineage_events.jsonl'}

### Deliverable 5 Summary

This deliverable implements the quality and lineage layer.

The quality gate uses a Great Expectations-style validation suite with expectations such as:

- required columns must exist,
- ticket IDs must not be null,
- ticket IDs must be unique,
- ticket subject and description must not be empty,
- standardized priority and channel values must be valid,
- Gold knowledge documents must contain usable content,
- Gold layer must not include PII columns such as customer name or email.

The pipeline also emits OpenLineage-style JSON events for the main jobs:

- ingestion,
- lakehouse storage,
- RAG pipeline,
- orchestration,
- quality gate.

The emitted events are saved as JSONL under the lineage output folder.

# Final Project Files
## Summary, requirements, README, and final verification

In [63]:
# FINAL — Capstone Deliverables Summary

ordered_dag_tasks = [
    log["task_id"] for log in dag_report["execution_log"]
]

final_project_summary = {
    "project_title": "Real-Time Customer Support Intelligence Platform",
    "dataset": KAGGLE_DATASET,
    "environment": "Local JupyterLab on Windows",
    "deliverables": {
        "Deliverable 1": {
            "name": "Ingestion Layer",
            "status": "completed",
            "components": [
                "Mock Kafka producer",
                "Mock Kafka consumer",
                "Pydantic schema validation",
                "Valid and invalid JSONL event outputs"
            ],
            "outputs": [
                str(VALID_EVENTS_FILE),
                str(INVALID_EVENTS_FILE)
            ]
        },
        "Deliverable 2": {
            "name": "Delta Lakehouse Simulation",
            "status": "completed",
            "components": [
                "Bronze zone",
                "Silver zone",
                "Gold zone",
                "Schema enforcement",
                "MERGE / UPSERT simulation"
            ],
            "outputs": [
                str(BRONZE_FILE),
                str(MERGED_SILVER_FILE),
                str(MERGED_GOLD_FILE)
            ]
        },
        "Deliverable 3": {
            "name": "RAG Pipeline",
            "status": "completed",
            "components": [
                "Chunking",
                "SentenceTransformer embeddings",
                "Vector search",
                "BM25 keyword search",
                "Hybrid search with RRF",
                "Cross-encoder reranking"
            ],
            "outputs": [
                str(CHUNKS_FILE),
                str(RAG_RESULTS_FILE)
            ]
        },
        "Deliverable 4": {
            "name": "Orchestration",
            "status": "completed",
            "components": [
                "Airflow-style DAG runner",
                "Task dependencies",
                "End-to-end DAG execution report",
                "Reference Airflow DAG file"
            ],
            "tasks_executed_in_order": ordered_dag_tasks,
            "outputs": [
                str(DAG_RUN_REPORT_FILE),
                str(AIRFLOW_DAG_FILE)
            ]
        },
        "Deliverable 5": {
            "name": "Quality Gate and OpenLineage",
            "status": "completed",
            "components": [
                "Great Expectations-style validation suite",
                "Quality report",
                "Failed records output",
                "OpenLineage-style JSONL events"
            ],
            "outputs": [
                str(GE_SUITE_FILE),
                str(QUALITY_REPORT_FILE),
                str(FAILED_QUALITY_FILE),
                str(OPENLINEAGE_EVENTS_FILE)
            ]
        }
    }
}

FINAL_SUMMARY_FILE = BASE_DIR / "outputs" / "final_project_summary.json"

with open(FINAL_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(final_project_summary, f, indent=2, ensure_ascii=False, default=str)

final_project_summary

{'project_title': 'Real-Time Customer Support Intelligence Platform',
 'dataset': 'suraj520/customer-support-ticket-dataset',
 'environment': 'Local JupyterLab on Windows',
 'deliverables': {'Deliverable 1': {'name': 'Ingestion Layer',
   'status': 'completed',
   'components': ['Mock Kafka producer',
    'Mock Kafka consumer',
    'Pydantic schema validation',
    'Valid and invalid JSONL event outputs'],
   'outputs': ['C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\valid_events.jsonl',
    'C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\ingestion\\invalid_events.jsonl']},
  'Deliverable 2': {'name': 'Delta Lakehouse Simulation',
   'status': 'completed',
   'components': ['Bronze zone',
    'Silver zone',
    'Gold zone',
    'Schema enforcement',
    'MERGE / UPSERT simulation'],
   'outputs': ['C:\\Users\\alamr\\OneDrive\\Course\\MDE for AI Systems\\Capstone_Project\\outputs\\bronze\\support_tickets

In [54]:
# FINAL — Create requirements.txt

requirements_text = """
pandas
numpy
pyarrow
pydantic
loguru
kagglehub
sentence-transformers
scikit-learn
rank-bm25
chromadb
""".strip()

REQUIREMENTS_FILE = BASE_DIR / "requirements.txt"

with open(REQUIREMENTS_FILE, "w", encoding="utf-8") as f:
    f.write(requirements_text)

print("requirements.txt created:")
print(REQUIREMENTS_FILE)

requirements.txt created:
C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\requirements.txt


In [55]:
# FINAL — Create README.md

README_FILE = BASE_DIR / "README.md"

readme_text = f"""
# Real-Time Customer Support Intelligence Platform

## Project Overview

This capstone project implements a real-time customer support intelligence platform for modern data engineering and AI systems.

The platform ingests customer support tickets, validates their schema, stores them in a lakehouse-style architecture, builds a RAG pipeline over the curated knowledge base, orchestrates the workflow using an Airflow-style DAG, and emits quality and lineage reports.

## Dataset

The project uses the Kaggle Customer Support Ticket Dataset:

`{KAGGLE_DATASET}`

PII columns such as customer name and customer email are excluded from the Gold knowledge base and RAG pipeline.

---

## Architecture

The project pipeline includes five main layers:

1. Ingestion Layer
2. Delta Lakehouse Simulation
3. RAG Pipeline
4. Orchestration Layer
5. Quality and Lineage Layer

---

## Deliverable 1 — Ingestion Layer

Components:

- Mock Kafka producer
- Mock Kafka consumer
- Pydantic schema validation
- Valid and invalid event outputs

Outputs:

- `outputs/ingestion/valid_events.jsonl`
- `outputs/ingestion/invalid_events.jsonl`

Execution result:

- Produced events: 50
- Valid events: 50
- Invalid events: 0

---

## Deliverable 2 — Delta Lakehouse Simulation

This deliverable implements a lightweight Delta Lakehouse simulation using Parquet files.

Components:

- Bronze zone
- Silver zone
- Gold zone
- Schema enforcement
- MERGE / UPSERT simulation

Execution result:

- Bronze records: 50
- Silver records before merge: 50
- Silver records after merge: 51
- Gold documents before merge: 18
- Gold documents after merge: 20
- Updated records: 1
- Inserted records: 1
- Schema enforcement: passed

Outputs:

- `outputs/bronze/support_tickets_bronze.parquet`
- `outputs/silver/support_tickets_silver_merged.parquet`
- `outputs/gold/support_knowledge_base_gold_merged.parquet`

---

## Deliverable 3 — RAG Pipeline

Components:

- Chunking
- SentenceTransformer embeddings
- Vector search using cosine similarity
- BM25 keyword search
- Hybrid search using Reciprocal Rank Fusion
- Cross-encoder reranking

Execution result:

- Gold documents: 20
- Chunks created: 36
- Embedding model: `all-MiniLM-L6-v2`
- Reranking model: `cross-encoder/ms-marco-MiniLM-L-6-v2`

Test query:

`How can I solve a product setup issue?`

Outputs:

- `outputs/rag/rag_chunks.parquet`
- `outputs/rag/rag_search_results.json`

---

## Deliverable 4 — Orchestration

The DAG connects the main modules end-to-end:

1. Ingestion
2. Lakehouse storage
3. RAG pipeline

A reference Airflow DAG file is generated under the `dags/` folder.

Execution result:

- DAG status: success
- Tasks executed:
  - task_1_ingestion
  - task_2_lakehouse_storage
  - task_3_rag_pipeline

Outputs:

- `outputs/orchestration/dag_run_report.json`
- `dags/customer_support_intelligence_dag.py`

---

## Deliverable 5 — Quality Gate and OpenLineage

Components:

- Great Expectations-style validation suite
- Quality report
- Failed quality records output
- OpenLineage-style JSONL events

Execution result:

- Quality gate passed: True
- Total expectations: 9
- Passed expectations: 9
- Failed expectations: 0
- OpenLineage events emitted: 10

Outputs:

- `outputs/quality/great_expectations_suite.json`
- `outputs/quality/quality_report.json`
- `outputs/quality/failed_quality_records.parquet`
- `outputs/lineage/openlineage_events.jsonl`

---

## How to Run

Install dependencies:

```bash
pip install -r requirements.txt
```

Run JupyterLab:

```bash
jupyter lab
```

Then open the notebook and run all cells from top to bottom.

---

## Project Notes

This project is designed as a two-day educational capstone.

Simulated components:

- Kafka is simulated using a Python `MockKafka` class.
- Delta Lake is simulated using Parquet zones with schema enforcement and MERGE-like logic.
- Airflow is simulated using an Airflow-style DAG runner.
- Great Expectations is simulated using expectation-style checks.
- OpenLineage is simulated by emitting JSONL lineage events.

The RAG pipeline uses real embedding, retrieval, hybrid search, and reranking components.

---

## Repository Structure

```text
customer_support_intelligence_platform/
│
├── README.md
├── requirements.txt
├── capstone_project_organized.ipynb
│
├── dags/
│   └── customer_support_intelligence_dag.py
│
├── outputs/
│   ├── ingestion/
│   ├── bronze/
│   ├── silver/
│   ├── gold/
│   ├── rag/
│   ├── orchestration/
│   ├── quality/
│   ├── lineage/
│   └── final_project_summary.json
```

---

## Author

Reem Alamri
""".strip()

with open(README_FILE, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README.md created:")
print(README_FILE)

README.md created:
C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\README.md


In [57]:
# FINAL — Check Important Files

important_files = {
    "README": README_FILE,
    "Requirements": REQUIREMENTS_FILE,
    "Final Project Summary": FINAL_SUMMARY_FILE,
    "Valid Events": VALID_EVENTS_FILE,
    "Bronze Table": BRONZE_FILE,
    "Merged Silver Table": MERGED_SILVER_FILE,
    "Merged Gold Table": MERGED_GOLD_FILE,
    "RAG Chunks": CHUNKS_FILE,
    "RAG Results": RAG_RESULTS_FILE,
    "DAG Run Report": DAG_RUN_REPORT_FILE,
    "GE Suite": GE_SUITE_FILE,
    "Quality Report": QUALITY_REPORT_FILE,
    "Failed Quality Records": FAILED_QUALITY_FILE,
    "OpenLineage Events": OPENLINEAGE_EVENTS_FILE,
    "Airflow DAG File": AIRFLOW_DAG_FILE,
}

file_check = []

for name, file_path in important_files.items():
    file_check.append({
        "name": name,
        "file": str(file_path),
        "exists": Path(file_path).exists()
    })

file_check_df = pd.DataFrame(file_check)

pd.set_option("display.max_colwidth", None)
file_check_df

,name,file,exists
0,README,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\README.md,True
1,Requirements,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\requirements.txt,True
2,Final Project Summary,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\final_project_summary.json,True
3,Valid Events,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\ingestion\valid_events.jsonl,True
4,Bronze Table,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\bronze\support_tickets_bronze.parquet,True
5,Merged Silver Table,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\silver\support_tickets_silver_merged.parquet,True
6,Merged Gold Table,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\gold\support_knowledge_base_gold_merged.parquet,True
7,RAG Chunks,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\rag\rag_chunks.parquet,True
8,RAG Results,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\rag\rag_search_results.json,True
9,DAG Run Report,C:\Users\alamr\OneDrive\Course\MDE for AI Systems\Capstone_Project\outputs\orchestration\dag_run_report.json,True
